In [1]:
import os
import datetime
import warnings
import json
import polars as pl
import pandas as pd
import altair as alt

from src.najdi_rok import najdi_rok
from src.pocet_stran import pocet_stran
from src.bez_bordelu import bez_bordelu
from src.alt_friendly import alt_friendly
from src.hezke_jmeno import hezke_jmeno
from src.kristi_promin import kristi_promin
from src.zjisti_vazbu import zjisti_vazbu
from src.me_to_neurazi import me_to_neurazi

pl.Config(tbl_rows=100)
alt.data_transformers.disable_max_rows()
alt.themes.register('irozhlas', kristi_promin)
alt.themes.enable('irozhlas')
warnings.filterwarnings('ignore')

In [2]:
with open(os.path.join('src','kredity.json'), 'r', encoding='utf-8') as kredity:
    kredity = json.loads(kredity.read())

In [3]:
with open(os.path.join('barvy','barvy.json'), 'r', encoding='utf-8') as palety:
    palety = json.loads(palety.read())

In [4]:
df = pl.read_parquet('data/cnb_vyber.parquet')

In [5]:
df = df.with_columns(pl.col('008').map_elements(najdi_rok, return_dtype=int).alias('rok'))

In [6]:
petdvanula = pl.read_parquet("data/cnb_sloupce/520.parquet")
df = df.join(petdvanula, on='001', how='left')

In [7]:
pl.read_parquet("data/cnb_sloupce/041.parquet")

041_ind1,041_a,041_h,041_b,041_k,041_g,041_f,041_d,041_e,041_j,041_n,041_m,001
str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,str
"""1""","[""eng""]","[""und""]",null,null,null,null,null,null,null,null,null,"""ck8300094"""
"""1""","[""cze""]","[""rus""]",null,null,null,null,null,null,null,null,null,"""ck8300101"""
"""1""","[""cze""]","[""slo""]",null,null,null,null,null,null,null,null,null,"""ck8300109"""
"""0""","[""cze"", ""rus""]",null,null,null,null,null,null,null,null,null,null,"""ck8300112"""
"""1""","[""cze""]","[""slo""]",null,null,null,null,null,null,null,null,null,"""ck8300119"""
"""1""","[""pol""]","[""cze""]",null,null,null,null,null,null,null,null,null,"""ck8300122"""
"""1""","[""cze""]","[""slo""]",null,null,null,null,null,null,null,null,null,"""ck8300123"""
"""1""","[""pol"", ""rus""]","[""cze""]",null,null,null,null,null,null,null,null,null,"""ck8300124"""
"""0""","[""cze"", ""pol""]",null,null,null,null,null,null,null,null,null,null,"""ck8300134"""


In [8]:
df = df.join(pl.read_parquet("data/cnb_sloupce/041.parquet"), on="001", how="left")

In [9]:
df = df.explode("041_h").unique(subset=["rok","100_a","245_a","041_h"],keep="first")

In [10]:
df = df.with_columns(
    pl.when(
        ((pl.col('041_h').is_null()) | (pl.col('041_h') == 'cze'))
    ).then(
        pl.lit('původní česká')).otherwise(
            pl.lit("překladová")
    ).alias("preklad"))

In [11]:
tematicke_sloupce = ['072_x','245_a','245_c','245_p','246_a','500_a','520_a','520_b','521_a','648_a','650_a','650_x','650_z','650_y','651_a','653_a','655_a','964_a']

In [12]:
df = df.select(pl.col(['rok','preklad'] + tematicke_sloupce)).with_columns(pl.col(tematicke_sloupce).cast(pl.List(pl.String)).list.join(" "))
df = df.with_columns(pl.col(tematicke_sloupce).fill_null(' ')).with_columns(pl.concat_str([pl.col(t) for t in tematicke_sloupce], separator=" ").alias("keywords"))

In [13]:
def tema(label, keywords):
    return df.filter(pl.col('keywords').str.contains(keywords)).with_columns(téma = pl.lit(label)).with_columns(
               pl.col("rok").map_elements(
                   lambda x: datetime.date(year=x, month=1, day=1), 
                   return_dtype=pl.Date
               ).cast(pl.Datetime)
           )

In [14]:
warnings.filterwarnings("ignore")

In [15]:
def hledej_tema(label, keywords):
    return df.filter(pl.col('rok') > 1945).filter(pl.col('keywords').str.contains(keywords)).with_columns(téma = pl.lit(label)).with_columns(
               pl.col("rok").map_elements(
                   lambda x: datetime.date(year=x, month=1, day=1), 
                   return_dtype=pl.Date
               ).cast(pl.Datetime)
           )

In [16]:
def hledej_tema(label, keywords):
        return df.filter(pl.col('keywords').str.contains(keywords)).with_columns(téma = pl.lit(label))

In [17]:
predpo = df.filter(
    pl.col("rok").is_between(1987,1993)
).with_columns(
    pl.col("keywords").map_elements(lambda x: x.split(" "))
).explode("keywords")

In [18]:
predpo.filter(
    pl.col("rok").is_between(1987,1988)
).group_by('keywords').len().join(
    predpo.filter(
        pl.col("rok").is_between(1990,1992)
    ).group_by('keywords').len(), on='keywords', how='right'
).with_columns((pl.col('len_right') / pl.col('len')).alias('narust')).filter(pl.col('len_right') > 20).filter(pl.col('narust').is_null()).sort(by='len_right',descending=True).head(100)

len,keywords,len_right,narust
u32,str,u32,f64
null,"""sčítání""",3284,null
null,"""bytů""",1252,null
null,"""Sčítání""",696,null
null,"""1991.""",665,null
null,"""domy,""",467,null
null,"""Obyvatelstvo,""",466,null
null,"""domácnosti.""",465,null
null,"""střed.""",283,null
null,"""ČSFR""",191,null


In [19]:
hledej_tema("povinnosti","povin")

rok,preklad,072_x,245_a,245_c,245_p,246_a,500_a,520_a,520_b,521_a,648_a,650_a,650_x,650_z,650_y,651_a,653_a,655_a,964_a,keywords,téma
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
1950,"""původní česká""",""" ""","""Lisař Kubát :""","""Jiří Beneš""",""" """,""" ""","""25000 výt.""","""Dělník Pavel Kubát je uvědoměl…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Budování socialismu - rodina -…",""" Lisař Kubát : Jiří Beneš …","""povinnosti"""
1951,"""překladová""",""" ""","""Borjovi noví přátelé /""","""I. Grinberg ; z rus. pův. vyd.…",""" """,""" ""","""20000 výt.""","""Mateřská škola, výchova dětí, …",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Škola mateřská sovětská - vypr…",""" Borjovi noví přátelé / I. Gr…","""povinnosti"""
1973,"""původní česká""",""" ""","""Vyhláška č. 80 z 20.10.1966 ;""",""" """,""" """,""" ""","""Obálka: Ota Karlas Zdobený tit…","""Vyhláška o pravidlech silniční…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Provoz silniční - pravidla Dop…",""" Vyhláška č. 80 z 20.10.1966 …","""povinnosti"""
2022,"""překladová""","""Americká próza Literatura pro …","""Polibek bez duše /""","""Jennifer L. Armentrout ; z ang…",""" """,""" ""","""Na obálce pod názvem: série Da…","""Young adult fantasy román vypr…","""Layla je napůl Strážce, napůl …",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""americké romány fantasy romány…",""" ""","""Americká próza Literatura pro …","""povinnosti"""
2021,"""překladová""","""Německá próza, německy psaná L…","""Kroniky prachu /""","""Lin Rina ; z německého originá…",""" """,""" ""","""Údaj o vydání je chybný, správ…","""Dívčí román německé autorky, z…","""Šaty, plesy, plané řeči a na k…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""německé romány literatura youn…",""" ""","""Německá próza, německy psaná L…","""povinnosti"""
1951,"""překladová""",""" ""","""Kačátko :""","""N. Gernětová aT. Gurevičová ; …",""" """,""" ""","""8800 výt.""","""Hra, uvedená po prvé v moskev.…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Hry maňáskové""",""" Kačátko : N. Gernětová aT. G…","""povinnosti"""
1942,"""překladová""",""" ""","""Účetnictví v maloobchodě dle p…","""napsal Wolfgang Foerster ; pře…",""" """,""" ""","""8451-11450 výt.""",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" Účetnictví v maloobchodě dle…","""povinnosti"""
1938,"""původní česká""","""Vojenství. Obrana země. Ozbroj…","""Zprošťování od činné vojenské …","""[zpracovalo hl. št. 1. oddělen…",""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""mobilizace (vojenství) branná …",""" ""","""Česko Česko""",""" """,""" """,""" ""","""předpisy""",""" ""","""Vojenství. Obrana země. Ozbroj…","""povinnosti"""
2022,"""překladová""","""Německá próza, německy psaná""","""Někdo zůstává /""","""Laura Kneidl ; z německého ori…",""" """,""" """,""" ""","""Milostný román německé autorky…","""Aliza by se ráda zamilovala, a…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""německé romány milostné romány…",""" ""","""Německá próza, německy psaná N…","""povinnosti"""


In [43]:
revoluce = {
    'povinnosti': '[Pp]ovinnost',
    'Ježíš': '[Jj]ežíš',
#    'volný čas': '[Vv]olný čas',
#    'přestavba': '[Pp]řestavb',
    'horoskopy': '([Hh]oroskop|[Zz]věrokru|[Aa]strolo)',
    'privatizace': '[Pr]ivatiza',
#    'kolektiv': '[Kk]olektiv',
#    'soukromník': '[Ss]oukrom(ý|ní)',
    'produktivita': '[Pp]roduktivi',
    'pionýr': ('[Pi]oný[rř]'),
    'skaut, junák': '([Ss]kaut|[Jj]uná[kc])',
    'spartakiáda': '[Ss]partakiá',
    'sokolský slet': '[Ss]okols\w{1,5}\sslet',
#    'solidarita': '[Ss]olid[áa]r',
#    'úspěchy': '[Úú]spě',
#    'demokracie': '[Dd]emokra',
#    'tyranie':'[Tt]yran',
    'RVHP': '(RVHP|Rad\w{1,5} vzájemné hospodářské pomoci)',
    'ES/EU':'(Evropsk\w{1,3}\suni\w{1,4}|Evropsk\w{1,3}\sspolečen)'
    
#    'solidarita': '[Ss]olid[aá]',
#    'zisk': '[Zz]isk',
#    'tradice':'[Tt]radi[cč]',
#    'sex': '[S]ex',
    #'Marx': '[Mm]arx',
    #'Masaryk': '[Mm]asaryk',
}

titulek = "Vybraná témata knih ve 20. století"
podtitulek = "Co tečka, to kniha: růžová původní, modrá překladová. (Včetně opakovaných vydání.)"

finalni_graf_data = pl.concat([hledej_tema(tema, retezec) for tema, retezec in revoluce.items()]).filter(pl.col("rok").is_between(1900,2000)).with_columns(
                   pl.col("rok").map_elements(
                       lambda x: datetime.date(year=x, month=1, day=1), 
                       return_dtype=pl.Date
                   ).cast(pl.Datetime)
               )
finalni_graf_data

temata_zlomy = alt.Chart(
    finalni_graf_data.to_pandas(), title=alt.Title(titulek,subtitle=podtitulek)
).mark_circle(size=6, opacity=1).encode(
    x=alt.X("rok:T", title=None, axis=alt.Axis(domainOpacity=0, tickColor='#ECEDE6', tickCount=8)), 
    y=alt.Y('téma:N', 
            title=None, 
            axis=alt.Axis(domainOpacity=0, tickColor='white'),
            sort=[tema for tema, neco in revoluce.items()]), 
            yOffset=alt.YOffset("jitter:Q", scale=alt.Scale(range=[2, 15])), 
            color=alt.Color('preklad:N', scale=alt.Scale(range=['#E09DA3','#81A9D5']), 
            sort=[tema for tema, retezec in revoluce.items()]
).legend(None)
).transform_calculate(jitter="sqrt(-2*log(random()))*cos(2*PI*random())")

rulecolor = '#81A9D5'

rule1 = alt.Chart(alt.Data(values=[{'rok': '1948-02-25'}])).mark_rule(
    color=rulecolor,
    strokeDash=[1.5, 4]  # Optional: makes the line dashed
).encode(
    x='rok:T'
)

rule2 = alt.Chart(alt.Data(values=[{'rok': '1989-11-17'}])).mark_rule(
    color=rulecolor,
    strokeDash=[1.5, 4]  # Optional: makes the line dashed
).encode(
    x='rok:T'
)

# Add text annotation
text1 = alt.Chart(alt.Data(values=[{'rok': '1948-02-25', 'y': 0.3}])).mark_text(
    # angle=270,  # Rotates text vertically
    align='right',
    baseline='middle',
    dy=70,# Slight horizontal offset from the line
    dx=-6.5,
    text=['komunistický','převrat →'],
    font='Asap',
    size=8
).encode(
    x='rok:T',
    y=alt.value(0)  # Places text at bottom of chart
)


# Add text annotation
text2 = alt.Chart(alt.Data(values=[{'rok': '1989-11-17', 'y': 0.3}])).mark_text(
    # angle=270,  # Rotates text vertically
    align='right',
    baseline='middle',
    dy=73,# Slight horizontal offset from the line
    dx=-6.5,
    text=['sametová revoluce →'],
    font='Asap',
    size=8
).encode(
    x='rok:T',
    y=alt.value(0)  # Places text at bottom of chart
)

# Add text annotation
text3 = alt.Chart(alt.Data(values=[{'rok': '1918-10-28', 'y': 0.3}])).mark_text(
    # angle=270,  # Rotates text vertically
    align='right',
    baseline='middle',
    dy=70,# Slight horizontal offset from the line
    dx=-6.6,
    text=['založení','ČSR →'],
    font='Asap',
    size=8
).encode(
    x='rok:T',
    y=alt.value(0)  # Places text at bottom of chart
)

rule3 = alt.Chart(alt.Data(values=[{'rok': '1918-10-28'}])).mark_rule(
    color=rulecolor,
    strokeDash=[1.5, 4]  # Optional: makes the line dashed
).encode(
    x='rok:T'
)

temata_zlomy_komb = (temata_zlomy + rule1 + rule2 + rule3 + text1 + text2 + text3).properties(
    width=kredity['sirka'] * 1.2, autosize={'type': 'fit', 'contains': 'padding'}
).configure_view(stroke='transparent').configure_axisX(grid=False, domain=False)

temata_zlomy_komb

alt.LayerChart(...)

In [45]:
me_to_neurazi(temata_zlomy_komb, kredity=kredity['default'], soubor="01_temata")

<figure>
    <a href="https://data.irozhlas.cz/knihy-grafy/01_temata.svg" target="_blank">
    <img src="https://data.irozhlas.cz/knihy-grafy/01_temata.svg" width="100%" alt="Omlouváme se, ale alternativní text se nepodařilo vygenerovat. Texty v grafu by měly být čitelné ze zdrojového souboru SVG." />
    </a>
    </figure>


## Hrajeme na přání

In [54]:
na_prani = { 'Tibet':'[Tt]ibet'
}

na_prani_data = pl.concat([hledej_tema(tema, retezec) for tema, retezec in na_prani.items()]).filter(pl.col("rok").is_between(1900,2000)).with_columns(
                   pl.col("rok").map_elements(
                       lambda x: datetime.date(year=x, month=1, day=1), 
                       return_dtype=pl.Date
                   ).cast(pl.Datetime)
               )

na_prani_graf = alt.Chart(
    na_prani_data.to_pandas(), title=alt.Title("Graf na přání"), width=kredity['sirka'] * 1.2
).mark_circle(size=6, opacity=1).encode(
    x=alt.X("rok:T", title=None, axis=alt.Axis(domainOpacity=0, tickColor='#ECEDE6', tickCount=8)), 
    y=alt.Y('téma:N', 
            title=None, 
            axis=alt.Axis(domainOpacity=0, tickColor='white'),
            sort=[tema for tema, neco in revoluce.items()]), 
            yOffset=alt.YOffset("jitter:Q", scale=alt.Scale(range=[2, 15])), 
            color=alt.Color('preklad:N', scale=alt.Scale(range=['#E09DA3','#81A9D5']), 
            sort=[tema for tema, retezec in revoluce.items()]
).legend(None)
).transform_calculate(jitter="sqrt(-2*log(random()))*cos(2*PI*random())")

na_prani_graf

alt.Chart(...)

In [60]:
hledej_tema('skaut, junák', '([Ss]kaut|[Jj]uná[kc])').filter(pl.col("rok").is_between(1949,1967)).sort(by="rok")

rok,preklad,072_x,245_a,245_c,245_p,246_a,500_a,520_a,520_b,521_a,648_a,650_a,650_x,650_z,650_y,651_a,653_a,655_a,964_a,keywords,téma
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
1949,"""původní česká""",""" ""","""Napucánek /""","""Josef Spilka ; Il. ... Zdeněk …",""" """,""" ""","""10750 výt. Barevné ilustrace""","""""Tak vám byl jednou jeden chla…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Zakarpatská Ukrajina - vyprávě…",""" Napucánek / Josef Spilka ; I…","""skaut, junák"""
1949,"""původní česká""",""" ""","""Táborový deník šestnáctiletého…","""Se slovesným přehlédnutím text…",""" """,""" ""","""7000 výt.""",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Wolker, Jiří (1900-1924 čes. b…",""" Táborový deník šestnáctileté…","""skaut, junák"""
1949,"""původní česká""",""" ""","""Vodáci z Olivové zátoky :""","""Adolf Veselý""",""" """,""" ""","""5500 výtisků""",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Moře Jaderské - knihy pro mlád…",""" Vodáci z Olivové zátoky : Ad…","""skaut, junák"""
1949,"""původní česká""",""" ""","""Zápisník časopisu Vpřed [na ro…","""Sest. red. kruh čas. Vpřed""",""" """,""" ""","""Notace Barevné ilustrace""",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Junáci - kalendáře Kalendáře""",""" Zápisník časopisu Vpřed [na …","""skaut, junák"""
1950,"""původní česká""",""" ""","""Pojďte s pionýry :""","""zpracoval redakční kruh za ved…",""" """,""" ""","""15000 výt.""",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Junáci - pionýři Pionýři POJ (…",""" Pojďte s pionýry : zpracoval…","""skaut, junák"""
1950,"""překladová""",""" ""","""Stará pevnost /""","""Vladimír Beljajev ; z rus. ori…",""" """,""" ""","""7700 výt.""","""""Stará pevnost"", první díl tri…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Ukrajina - boje s petljurovci …",""" Stará pevnost / Vladimír Bel…","""skaut, junák"""
1951,"""překladová""",""" ""","""Létající koráb a jiné pohádky …","""Alexander Nikolajevič Afanasje…",""" """,""" ""","""14750 výt. Barevně ilustrované…","""Celkem 16 tu známějších, tu mé…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Pohádky ruské""",""" Létající koráb a jiné pohádk…","""skaut, junák"""
1951,"""překladová""",""" ""","""O junákovi s planoucím srdcem …","""Z.G. Ljubina ; Z rus. pův. vyd…",""" """,""" ""","""12000 výt.""","""Symbolická pohádka o statečném…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Pohádky symbolické""",""" O junákovi s planoucím srdce…","""skaut, junák"""
1952,"""původní česká""",""" ""","""Korespondence s rodiči /""","""Jiří Wolker ; Uspoř. Zdena Wol…",""" """,""" ""","""5000 výt. Pozn.""","""Soubor 282 dopisů, jež přispěj…",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" ""","""Wolker, Jiří (1900-1924 básník…",""" Korespondence s rodiči / Jiř…","""skaut, junák"""


In [62]:
hledej_tema('skaut, junák', '([Ss]kaut|[Jj]uná[kc])').filter(pl.col("rok").is_between(1949,1967)).sort(by="rok").write_csv("data/skaut-junak.csv")